In [ ]:
import torch
import torch.nn as nn

In [ ]:

inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

### Simple self attention

In [ ]:
cntx_vector=torch.zeros(inputs.shape[0],inputs.shape[1])
print(cntx_vector)        

In [ ]:
for i,inp in enumerate(inputs):
    for j,inpi in enumerate(torch.softmax(inp @ torch.transpose(inputs,1,0),dim=0)):
        cntx_vector[i] +=inpi*inputs[j]

In [ ]:
print(cntx_vector)

### Self attention using learnable weights

In [ ]:
d_out=8
d_model=inputs.shape[1]
torch.manual_seed(123)
wq=nn.Linear(d_model,d_out,bias=False)
wk=nn.Linear(d_model,d_out,bias=False)
wv=nn.Linear(d_model,d_out,bias=False)

In [ ]:
# Query, Key and Value
Q=wq(inputs)
K=wk(inputs)
V=wv(inputs)
#Attention Score
att_score=Q@(K.T)
#attention weight
att_weight=torch.softmax(att_score/d_out**0.5,dim=-1)
print(att_weight.shape)
#Context vector
context_vactor=att_weight@V
print(context_vactor)

### Casual attention mask

In [ ]:
wq=nn.Linear(d_model,d_out,bias=False)
wk=nn.Linear(d_model,d_out,bias=False)
wv=nn.Linear(d_model,d_out,bias=False)
# Query, Key and Value
Q=wq(inputs)
K=wk(inputs)
V=wv(inputs)
#Attention Score
att_score=Q@(K.T)
casual_mask=torch.tril(torch.ones(att_score.shape))
casual_mask=casual_mask.masked_fill(casual_mask==0,-torch.inf)
print(casual_mask)
print(att_score)
masked_att_score=att_score + casual_mask
print(masked_att_score)
att_weight=torch.softmax(masked_att_score/d_out**0.5,dim=-1)
print(att_weight)
#attention weight
#Context vector
context_vactor=att_weight@V
print(context_vactor)

In [ ]:
class causal_attention(nn.Module):
    def __init__(self,d_in,d_out,qkv_bias):
        super().__init__()
        #torch.manual_seed(123)
        self.W_query=nn.Linear(d_in,d_out,qkv_bias)
        self.W_key=nn.Linear(d_in,d_out,qkv_bias)
        self.W_value=nn.Linear(d_in,d_out,qkv_bias)
    def forward(self,x):
        query=self.W_query(x)
        key=self.W_key(x)
        value=self.W_value(x)
        #Attention Score
        attention_score=query @ (key.T)
        casual_mask=torch.tril(torch.ones(attention_score.shape))
        casual_att_score=casual_mask.masked_fill(casual_mask==0,-torch.inf)
        #attention weights
        attention_weight=torch.softmax(casual_att_score/(d_out**0.5),dim=-1)
        #context vector
        context_vector=attention_weight @ value
        return context_vector       

In [ ]:
self_att=causal_attention(inputs.shape[1],8,False)
print(self_att(inputs))

### Simple multihead attention

In [ ]:
class SimpleMultiheadAttention(nn.Module):
    def __init__(self,d_in,d_out,qkv_bias,n_heads):
        super().__init__()
        self.heads=nn.ModuleList([causal_attention(d_in,d_out,qkv_bias) for _ in range(n_heads)])
    def forward(self,x):
        return torch.concat([head(x) for head in self.heads],dim=-1)

In [ ]:
torch.manual_seed(123)
simpmultihead=SimpleMultiheadAttention(inputs.shape[1],8,False,2)
#Multihead attention context vector
torch.set_printoptions(sci_mode=False, precision=4)
print(simpmultihead(inputs))

#### Multihead Attention using weight split

In [ ]:
class causal_attention(nn.Module):
    def __init__(self,d_in,d_out,qkv_bias,n_heads):
        super().__init__()
        #torch.manual_seed(123)
        if d_out%num_heads!=0:
            assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.head_dim=d_out//num_heads
        self.num_heads=n_heads
        self.context_length=d_in


        self.W_query=nn.Linear(d_in,d_out,qkv_bias)
        self.W_key=nn.Linear(d_in,d_out,qkv_bias)
        self.W_value=nn.Linear(d_in,d_out,qkv_bias)
    def forward(self,x):
        query=self.W_query(x).view(self.context_length,self.num_heads,self.head_dim)
        key=self.W_key(x).view(self.context_length,self.num_heads,self.head_dim)
        value=self.W_value(x).view(self.context_length,self.num_heads,self.head_dim)
        query=torch.transpose(query,1,0)
        key=torch.transpose(key,1,0)
        value=torch.transpose(value,1,0)
        #Attention Score
        attention_score=query @ (key.T)
        casual_mask=torch.tril(torch.ones(attention_score.shape))
        casual_att_score=casual_mask.masked_fill(casual_mask==0,-torch.inf)
        #attention weights
        attention_weight=torch.softmax(casual_att_score/(d_out**0.5),dim=-1)
        #context vector
        context_vector=attention_weight @ value
        return context_vector       